# Pandas Analysis

学习用 Pandas 读取、筛选、分组和合并表格，处理文本、缺失值与时间，
并在 DataFrame 与 SQL 数据库之间读写数据。

**数据与范围：** `tutorials/data/posts_two_weeks.parquet`，
按质量标记筛选四川 2022-08-12 至 2022-08-25 的可用记录。
每行是一条采集到的帖子。用户 ID 已匿名化，文本已脱敏，位置经过网格化。
统计只描述这些采集记录；原采集覆盖未知，不能推广为全体用户的使用情况。
第 7 章使用微型表格说明脏值处理，该表不参与微博数据统计。

**运行准备：** 在 VS Code 打开整个项目，按 `requirements/environment.yml`
配置 `user-behavior-class` 环境，并选择对应的 Conda 内核。
用运行按钮或 `Shift+Enter` 执行当前单元格；重新启动内核后从头运行。
结果文件保存到 `data/output/pandas_tutorial/`。

### 目录

- [导入与环境](#setup)
- [1．读入文件：这张表里有什么？](#chapter1)
- [2．选取数据：哪些城市的记录最多？](#chapter2)
- [3．条件筛选：数量和比例有什么区别？](#chapter3)
- [4．分组统计：从帖子表到用户表](#chapter4)
- [5．合并与保存：把多张表组织起来](#chapter5)
- [6．字符串与重采样：讨论内容随日期怎样变化？](#chapter6)
- [7．清洗：识别缺失值、脏字符串与重复记录](#chapter7)
- [8．时间：字符串、Unix 时间戳和本地时区](#chapter8)
- [9．SQL：从数据库读表、把表写入数据库](#chapter9)
- [操作速查](#reference)

<a id="setup"></a>
## 开场：先准备工作台

`import pandas as pd` 给 Pandas 起一个短名字。`pd.read_parquet(...)` 表示调用它的读表函数。
`display(...)` 在 Notebook 中显示表格；`print(...)` 显示简短文字。
数据在 `tutorials/data/`。内核从 Notebook 目录启动时，相对路径是 `data/posts_two_weeks.parquet`；
如果 VS Code 从项目根目录启动，则使用 `tutorials/data/posts_two_weeks.parquet`。
下方自动兼容这两个位置，无需填写电脑上的绝对路径。

In [ ]:
from pathlib import Path
from io import StringIO
import sys
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display

assert (Path(sys.prefix) / 'conda-meta').exists(), '请在右上角选择 Conda 内核'
TUTORIAL_DIR = Path('.') if Path('data/posts_two_weeks.parquet').is_file() else Path('tutorials')
DATA = TUTORIAL_DIR / 'data/posts_two_weeks.parquet'
assert DATA.is_file(), '请从 tutorials 或项目根目录启动内核，并保留 tutorials/data'
ROOT = TUTORIAL_DIR.resolve().parent
OUT = ROOT / 'data/output/pandas_tutorial'
OUT.mkdir(parents=True, exist_ok=True)

# 自动选择本机已有的中文字体；没有时图题回退到英文，中文正文不受影响。
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
CJK_FONT = next((name for name in ['Noto Sans CJK SC', 'Arial Unicode MS',
               'Microsoft YaHei', 'SimHei', 'Songti SC', 'Heiti TC', 'Noto Sans CJK JP']
               if name in available_fonts), None)
if CJK_FONT:
    plt.rcParams['font.family'] = CJK_FONT
plt.rcParams.update({'figure.figsize': (9, 3.6), 'font.size': 11, 'axes.unicode_minus': False})
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 8)
pd.set_option('display.max_colwidth', 45)
print('Pandas 版本：', pd.__version__)
print('当前 Python：', sys.executable)
print('相对数据路径：', DATA)
print('中文绘图字体：', CJK_FONT or '未找到，图题将使用英文')

<a id="chapter1"></a>
## 1．读入文件：这张表里有什么？

### 1.1 先读成 DataFrame

**DataFrame 是带行索引、列名和列类型的二维表。** `df` 只是我们给这个对象起的变量名。
Parquet 保存表格的类型；`columns` 指定需要的列，`filters` 指定本课的记录范围。
这里沿用本项目的 `usable_text` 质量标记，其含义见 `tutorials/data/README.md`。
本例用 Pandas 直接读表，不依赖主课 Notebook 中的辅助函数。

In [ ]:
columns = ['post_id', 'user_id', 'created_at', 'date', 'city', 'city_code', 'text']
df = pd.read_parquet(DATA, columns=columns,
                     filters=[('province_code', '==', '51'), ('usable_text', '==', True)])
df['created_at'] = pd.to_datetime(df['created_at'], utc=True).dt.tz_convert('Asia/Shanghai')
print('行数、列数：', df.shape)
print('不同用户数：', df['user_id'].nunique())
print('观测到的日期数：', df['date'].nunique())
display(df.head(3))
assert len(df) > 0 and df['post_id'].is_unique

**怎么看结果：** `shape` 的两个数分别是行数和列数；在本数据上，筛选后有 54,609 条记录，
对应 36,463 个匿名用户和 14 个日期。`head(3)` 只展示前三行，原表仍保留全部记录。
同一用户可占多行，因此“帖子数”和“用户数”是两个不同单位。

### 1.2 列名与类型也要看

字符串适合存文本和 ID，时间类型支持日期运算。索引负责标记行，它不是帖子 ID。
`info()` 汇总非空数量与类型；`dtypes` 返回每列的类型。

In [ ]:
print('列名：', df.columns.tolist())
display(df.dtypes.to_frame('列类型'))
df.info()

### 1.3 CSV：分隔符、编码和类型为什么要指定？

CSV 是纯文本表格，不会像 Parquet 一样完整保存列类型。
为了离线演示，我们将**真实源的前 8 行**暂存成内存中的 CSV，再读回来。
`StringIO` 让字符串像文件一样被读取，不会产生第二份输入数据集。

In [ ]:
csv_text = df[['post_id', 'user_id', 'date', 'city_code', 'text']].head(8).to_csv(
    index=False, sep=';')
csv_demo = pd.read_csv(StringIO(csv_text), sep=';',
                       dtype={'post_id': 'string', 'user_id': 'string', 'city_code': 'string'},
                       parse_dates=['date'])
display(csv_demo.head(2))
display(csv_demo.dtypes.to_frame('CSV 读入后的类型'))
assert len(csv_demo) == 8

`sep=';'` 表示列之间以分号分隔；默认分隔符是逗号。`dtype` 让 ID 和编码保持为字符串，
避免前导零丢失。`parse_dates` 将指定列解析成时间。上述 `date` 只有日期，没有时区。

对真实磁盘文件，常见写法是：
```python
table = pd.read_csv('自己的文件.csv', sep=',', encoding='utf-8',
                    dtype={'user_id': 'string'})
```
出现乱码要核对来源编码，不能只凭更换编码后“不报错”认定文本正确。
日期格式有歧义时，应按来源格式用 `pd.to_datetime(..., format=...)` 明确解析。
[官方说明：读取和写入表格](https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html)

<a id="homework1"></a>
### 第 1 部分作业：重新读入 CSV，并确认列类型

本部分的 `csv_text` 保存了真实帖子前 8 行的分号分隔 CSV。请重新读成 `p1_table`：

1. 使用正确的分隔符，将 `post_id`、`user_id`、`city_code` 保持为字符串。
2. 把 `date` 读成时间类型，显示前 5 行、行列数和列类型。
3. 用一句话解释：为什么用户 ID、城市编码不宜直接读成数字？

`StringIO` 已在开场导入；可回看本部分的 `read_csv` 示例。

**请在下面的空白 Python 代码格中完成；参考答案紧接在其后，默认折叠。**

<details><summary>展开第 1 部分参考答案与完整代码</summary><p><strong>参考思路：</strong> 核对分隔符、ID 的类型和日期解析。`head(5)` 只是显示前 5 行， 读入的表仍有 8 行、5 列。</p><p>ID 和编码用于标识或匹配，不用于加减运算；按字符串保存还能保留可能有意义的前导零。 日期类型才方便后续日期处理。这个 CSV 中的日期没有时区，不能把它直接当成 UTC 时间点。</p><pre><code class="language-python">p1_table = pd.read_csv(
    StringIO(csv_text), sep=&#x27;;&#x27;,
    dtype={&#x27;post_id&#x27;: &#x27;string&#x27;, &#x27;user_id&#x27;: &#x27;string&#x27;, &#x27;city_code&#x27;: &#x27;string&#x27;},
    parse_dates=[&#x27;date&#x27;],
)
display(p1_table.head(5))
print(&#x27;行数、列数：&#x27;, p1_table.shape)
display(p1_table.dtypes.to_frame(&#x27;列类型&#x27;))</code></pre></details>

<a id="chapter2"></a>
## 2．选取数据：哪些城市的记录最多？

### 2.1 一列和一张表的区别

`df['city']` 是 **Series**：带索引的一维数据。
`df[['city']]` 是仍然具有二维表结构的 **DataFrame**。
两个列名写在同一个列表中，就能选取两列。

In [ ]:
city_series = df['city']
city_table = df[['city']]
print('一列：', type(city_series).__name__, city_series.shape)
print('一列表：', type(city_table).__name__, city_table.shape)
display(df[['city', 'text']].head(3))

### 2.2 用 iloc 按位置取行，用 loc 按标签取行

`iloc[:3]` 取第 0、1、2 个位置，右端点不包含。
`loc[101:103]` 按索引标签取行，在这个有序索引例子中包含标签 103。
为了看清区别，我们复制真实源前三行，并将索引改成 101、102、103。

In [ ]:
index_demo = df[['city', 'text']].head(3).copy()
index_demo.index = [101, 102, 103]
display(index_demo)
print('按位置取第 2、3 行：')
display(index_demo.iloc[1:3])
print('按标签取 101 到 103：')
display(index_demo.loc[101:103, ['city']])
assert len(index_demo.iloc[1:3]) == 2 and len(index_demo.loc[101:103]) == 3

### 2.3 value_counts：先数数，再画图

`value_counts()` 统计各取值出现多少次，默认按频数从高到低排列、排除缺失值。
加上 `dropna=False` 可把缺失值作为一项列出来。本例的 `city` 没有缺失。
城市名称沿用源文件的拼音标签，如 `Chengdu`，没有重写城市归属。

In [ ]:
city_counts = df['city'].value_counts(dropna=False).rename('记录数')
display(city_counts.head(8).to_frame())
ax = city_counts.head(8).plot.bar()
ax.set_title('各城市采集记录数（前 8 项）' if CJK_FONT else 'Collected posts by city (top 8)')
ax.set_xlabel('来源城市标签' if CJK_FONT else 'Source city label')
ax.set_ylabel('记录数' if CJK_FONT else 'Collected posts')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()
assert city_counts.sum() == len(df)

**怎么看结果：** 柱形图对应的是帖子记录数量。数量受城市规模、采集覆盖及少数高频用户等因素影响，
不能仅凭这张图认定哪座城市的人“更爱发微博”。先看统计表，确认指标和单位，再看图。
[官方说明：选取子集](https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html)

<a id="homework2"></a>
### 第 2 部分作业：选取两列，并生成城市频数表

继续使用完整帖子表 `df`：

1. 取出前 3 行的 `city` 和 `text` 两列，保持结果为 DataFrame，命名为 `p2_preview`。
2. 统计城市标签的记录数，取数量最多的 5 项，做成一列名为 `n_posts` 的表 `p2_top5`。
   该表的索引保留城市标签。
3. 打印 `df['city']` 与 `df[['city']]` 的对象类型，解释二者的区别。

可使用本部分的 `iloc`、`value_counts`、`head` 和 `to_frame`。

**请在下面的空白 Python 代码格中完成；参考答案紧接在其后，默认折叠。**

<details><summary>展开第 2 部分参考答案与完整代码</summary><p><strong>参考思路：</strong> 列名列表选择二维表，`iloc[:3]` 按位置选择前三行。 `value_counts` 默认按频数降序排列，`head(5)` 选择前五项，`to_frame` 把 Series 转成一列表。</p><p>`df[&#x27;city&#x27;]` 是一维 Series；`df[[&#x27;city&#x27;]]` 是二维 DataFrame。城市频数数的是采集记录， 不是城市人口，也不是不同用户数。</p><pre><code class="language-python">p2_preview = df[[&#x27;city&#x27;, &#x27;text&#x27;]].iloc[:3]
display(p2_preview)
p2_top5 = df[&#x27;city&#x27;].value_counts().head(5).to_frame(&#x27;n_posts&#x27;)
display(p2_top5)
print(&#x27;单层选列：&#x27;, type(df[&#x27;city&#x27;]).__name__)
print(&#x27;列表选列：&#x27;, type(df[[&#x27;city&#x27;]]).__name__)</code></pre></details>

<a id="chapter3"></a>
## 3．条件筛选：数量和比例有什么区别？

### 3.1 一个条件就是一列 True / False

`df['city'].eq(city_name)` 逐行比较城市是否等于指定值，返回布尔 Series。
`df.loc[条件, 列名列表]` 保留条件为 True 的行，再选择要显示的列。

In [ ]:
city_name = df['city'].mode().iloc[0]
in_city = df['city'].eq(city_name)
print('示范城市：', city_name)
display(in_city.head(5).to_frame('是否属于示范城市'))
display(df.loc[in_city, ['city', 'created_at', 'text']].head(3))
print('符合条件的行数：', in_city.sum())

### 3.2 多条件：& 是同时满足，| 是满足任意一个，~ 是取反

本例找出“来自示范城市，且正文包含‘工作’”的记录。
每个比较条件应加括号；逐行组合条件使用 `&` 或 `|`，不能用 Python 的 `and` / `or`。
`str.contains(..., regex=False, na=False)` 做字面匹配，将缺失文本按不命中处理。

In [ ]:
has_work = df['text'].str.contains('工作', regex=False, na=False)
selected = df.loc[in_city & has_work, ['user_id', 'city', 'text']].copy()
display(selected.head(3))
print('示范城市且含“工作”：', len(selected))
print('示范城市或含“工作”：', (in_city | has_work).sum())
print('不在示范城市：', (~in_city).sum())

### 3.3 数量大，不一定比例高

分子是各城市含“工作”的帖子数；分母是各城市全部可用帖子数。
Series 做除法会**按索引标签对齐**，不是按当前排列位置对齐。
部分城市没有命中记录，分子表缺少其标签，所以先 `reindex` 补齐未命中计数。

In [ ]:
total_by_city = df['city'].value_counts()
work_by_city = df.loc[has_work, 'city'].value_counts().reindex(total_by_city.index, fill_value=0)
work_fraction = work_by_city / total_by_city
city_comparison = pd.DataFrame({'全部记录数': total_by_city,
                                '含工作记录数': work_by_city,
                                '含工作比例': work_fraction})
display(city_comparison.sort_values('全部记录数', ascending=False).head(8))
assert city_comparison['含工作比例'].between(0, 1).all()

`0.05` 表示约 5% 的该城市采集记录含有这个词，不是“5% 的居民在工作”。
没有关键词命中可以将该城市的**命中计数**设为 0；没有分母的比例应保留缺失。
字面命中只是一条可复核的规则，尚不能作为完整主题分类或情绪诊断。

### 3.4 新增列与赋值

对筛选结果继续修改时先 `.copy()`。对某些行赋值用一次 `.loc[条件, 列名] = 值`。
避免 `df[条件]['列'] = ...` 这样的链式赋值。

In [ ]:
marked = df[['post_id', 'city']].head(8).copy()
marked['是否示范城市'] = marked['city'].eq(city_name)
marked['显示标记'] = '其他城市'
marked.loc[marked['是否示范城市'], '显示标记'] = '示范城市'
display(marked.head(3))

<a id="homework3"></a>
### 第 3 部分作业：在指定城市中筛选学习相关词记录

示范城市已经保存在 `city_name` 中。请筛选“来自该城市，且正文包含‘学习’或‘考试’”的记录：

1. 将结果保存为 `p3_selected`，只保留 `user_id`、`city`、`text` 三列，并显示前 3 行。
2. 计算这些记录占该城市全部采集记录的比例。
3. 说明你的分子和分母，以及为什么这个比例不能解释为“正在学习的用户比例”。

两个词可以分别用 `str.contains(..., regex=False, na=False)` 匹配，再用 `|` 组合。

**请在下面的空白 Python 代码格中完成；参考答案紧接在其后，默认折叠。**

<details><summary>展开第 3 部分参考答案与完整代码</summary><p><strong>参考思路：</strong> 城市条件与内容条件用 `&amp;` 组合，两种词命中用 `|` 组合。 分子是该城市命中任意一个词的帖子数；分母是该城市全部帖子数。</p><p>两个词同时命中的帖子只计一次。这是文本关键词比例，不是对用户真实学习行为的识别。 一个用户可有多条帖子，文本也可能是引用、广告或转述。用 `.copy()` 保留独立结果。</p><pre><code class="language-python">p3_in_city = df[&#x27;city&#x27;].eq(city_name)
p3_has_study = (
    df[&#x27;text&#x27;].str.contains(&#x27;学习&#x27;, regex=False, na=False)
    | df[&#x27;text&#x27;].str.contains(&#x27;考试&#x27;, regex=False, na=False)
)
p3_selected = df.loc[p3_in_city &amp; p3_has_study, [&#x27;user_id&#x27;, &#x27;city&#x27;, &#x27;text&#x27;]].copy()
display(p3_selected.head(3))
p3_denominator = int(p3_in_city.sum())
p3_share = len(p3_selected) / p3_denominator if p3_denominator else float(&#x27;nan&#x27;)
print(&#x27;示范城市：&#x27;, city_name)
print(&#x27;命中记录数、城市全部记录数、比例：&#x27;, len(p3_selected), p3_denominator, p3_share)</code></pre></details>

<a id="chapter4"></a>
## 4．分组统计：从帖子表到用户表

### 4.1 先拆成组，再在每组里计算

`groupby('weekday')` 把相同星期几的行分在一起；`.size()` 计算每组行数。
上海时区的 `.dt.dayofweek` 返回周一 0 到周日 6。
`sort_index()` 按索引排列，`sort_values()` 按统计数值排列，两者用途不同。

In [ ]:
df = df.assign(weekday=df['created_at'].dt.dayofweek,
               text_length=df['text'].str.len(),
               has_work=df['text'].str.contains('工作', regex=False, na=False))
weekday_counts = df.groupby('weekday').size().reindex(range(7), fill_value=0)
weekday_names = ['周一', '周二', '周三', '周四', '周五', '周六', '周日']
weekday_table = weekday_counts.rename('记录数').to_frame()
weekday_table['星期'] = weekday_names
display(weekday_table)

**怎么看结果：** 这两周每个星期几各出现两次，因此可比较各星期几的总数。
换成其他窗口，某个星期几可能出现更多次，需要先除以对应日历天数。
我们用明确的研究窗口计算分母，而不是只数有记录的日期。

In [ ]:
calendar = pd.date_range('2022-08-12', '2022-08-25', freq='D')
weekday_days = pd.Series(calendar.dayofweek).value_counts().reindex(range(7), fill_value=0)
weekday_table['窗口内日历天数'] = weekday_days
weekday_table['每个日历日平均记录数'] = weekday_counts / weekday_days.where(weekday_days.gt(0))
display(weekday_table)
assert weekday_days.sum() == 14

### 4.2 agg：一次生成多个统计指标

以 `user_id` 分组，生成每个用户一行的表。
`agg(新列名=('原列名', '聚合方法'))` 同时说明输出字段、输入字段和计算方式。

| 方法 | 含义 | 在本例中 |
|---|---|---|
| `size` | 组内行数 | 采集到的帖子数 |
| `count` | 该列非空值数量 | 若文本缺失则不计入 |
| `nunique` | 不同非空值的数量 | 观测到发帖的日期数 |
| `mean` | 平均值 | 文本长度均值；布尔列均值是 True 的比例 |
| `min` / `max` | 最小 / 最大值 | 窗口内首末观测时间 |

In [ ]:
users = df.groupby('user_id').agg(
    n_posts=('post_id', 'size'),
    active_days=('date', 'nunique'),
    mean_text_length=('text_length', 'mean'),
    work_share=('has_work', 'mean'),
    first_seen=('created_at', 'min'),
    last_seen=('created_at', 'max'),
)
display(users.head(3))
display(users[['n_posts', 'active_days', 'work_share']].describe())
display(users.sort_values('n_posts', ascending=False).head(3))
assert len(users) == df['user_id'].nunique()
assert users['n_posts'].sum() == len(df)
assert users['active_days'].between(1, 14).all()

**怎么看结果：** `user_id` 成了这张表的索引。`reset_index()` 可把它变回普通列。
`first_seen` 是本窗口中第一次采集到该用户的发帖时间，不是注册时间。
`describe()` 的 `count`、`mean`、`std`、分位数等是对用户指标的汇总。

同一词命中比例还有两种权重：直接对帖子求均值，高频用户权重更大；
先对每个用户求比例再取均值，每位已观测用户权重相同。

In [ ]:
weighting = pd.Series({
    '帖子等权的工作词命中比例': df['has_work'].mean(),
    '用户等权的工作词命中比例': users['work_share'].mean(),
}, name='比例')
display(weighting.to_frame())
display(users.reset_index().head(2))

<a id="homework4"></a>
### 第 4 部分作业：自己生成一张用户统计表

使用 `df` 生成每个用户一行的表 `p4_users`，包含：

- `n_posts`：记录数。
- `active_days`：出现记录的不同日期数。
- `mean_text_length`：平均正文字符数，使用本部分已经创建的 `text_length` 列。
- `observed_day_share`：不同观测日期数除以窗口的 14 个日历日。

显示前 5 行，并打印用户表累计记录数和原帖子表行数。解释为什么 `active_days` 用 `nunique`，
而记录数用 `size`，以及这张表中的一行代表什么。

**请在下面的空白 Python 代码格中完成；参考答案紧接在其后，默认折叠。**

<details><summary>展开第 4 部分参考答案与完整代码</summary><p><strong>参考思路：</strong> 用用户 ID 分组，通过命名聚合定义字段，再增加比例列。 `size` 数组内行数，`nunique` 数不同日期；同一天多次发帖只增加记录数，不增加不同日期数。</p><p>输出一行是一位已观测用户。累计记录数应与帖子表行数一致。 观测日占比描述采集窗口中的发帖日期覆盖，不能解释为用户每天使用微博的概率。</p><pre><code class="language-python">p4_users = df.groupby(&#x27;user_id&#x27;).agg(
    n_posts=(&#x27;post_id&#x27;, &#x27;size&#x27;),
    active_days=(&#x27;date&#x27;, &#x27;nunique&#x27;),
    mean_text_length=(&#x27;text_length&#x27;, &#x27;mean&#x27;),
)
p4_users[&#x27;observed_day_share&#x27;] = p4_users[&#x27;active_days&#x27;] / 14
display(p4_users.head(5))
print(&#x27;用户数：&#x27;, len(p4_users))
print(&#x27;累计记录数：&#x27;, p4_users[&#x27;n_posts&#x27;].sum(), &#x27;帖子表行数：&#x27;, len(df))</code></pre></details>

[官方说明：分组统计](https://pandas.pydata.org/docs/getting_started/intro_tutorials/06_calculate_statistics.html)

<a id="chapter5"></a>
## 5．合并与保存：把多张表组织起来

### 5.1 concat：将同结构的记录上下拼接

Cookbook 用多个月的天气表演示拼接；这里把同源的两周记录分成两段，再拼回来。
`axis=0` 沿行方向拼接，`ignore_index=True` 生成新行索引。
拼接不会自动删除重复帖子；生产数据中仍需要检查业务主键。

In [ ]:
first_week = df.loc[df['created_at'].lt(pd.Timestamp('2022-08-19', tz='Asia/Shanghai'))].copy()
second_week = df.loc[df['created_at'].ge(pd.Timestamp('2022-08-19', tz='Asia/Shanghai'))].copy()
combined = pd.concat([first_week, second_week], axis=0, ignore_index=True)
print('第一段、第二段、合并：', len(first_week), len(second_week), len(combined))
assert len(combined) == len(df) and combined['post_id'].is_unique
assert set(combined['post_id']) == set(df['post_id'])

### 5.2 concat(axis=1)：按索引并排放统计列

这里的两列索引都是城市标签，即使排列不同，也会按城市对齐。
默认保留双方所有索引；某一侧缺少标签时，对应位置产生缺失值。

In [ ]:
city_statistics = pd.concat([
    total_by_city.rename('n_city_posts'),
    work_fraction.sort_index().rename('city_work_share'),
], axis=1)
display(city_statistics.head(4))
assert city_statistics['n_city_posts'].sum() == len(df)

### 5.3 merge：按共同字段匹配两张表

把城市级汇总接到少量帖子上：左表每行是帖子，右表每行是城市。
`how='left'` 保留左表所有行；`validate='many_to_one'` 检查右表城市键唯一，
防止重复键让帖子数意外膨胀；`indicator=True` 显示匹配是否成功。

In [ ]:
city_lookup = city_statistics.rename_axis('city').reset_index()
joined = df[['post_id', 'city']].head(8).merge(
    city_lookup, on='city', how='left', validate='many_to_one', indicator=True)
display(joined.head(4))
display(joined['_merge'].value_counts().to_frame('匹配行数'))
assert len(joined) == 8 and joined['_merge'].eq('both').all()

**怎么看结果：** `both` 表示左右都存在这个城市键，`left_only` 表示左表记录没有找到右表匹配。
城市总体的命中比例接到帖子上，仍然是城市指标，不等于该帖子或该用户的个人属性。
连接前也要检查空键：Pandas 可以将双方缺失的键匹配起来，与常见 SQL 的空值连接规则不同。

### 5.4 保存统计结果

`index=False` 不将行索引另写为一列；需要保存索引时先把它变成有名称的普通列。
CSV 的 `utf-8-sig` 便于部分 Excel 版本识别中文编码。
下面保存的是城市汇总，不导出帖子正文，也不改写原始输入文件。
CSV 回读后，普通整数与可空整数等内部类型可能不同；本例检查保存前后的值一致。

In [ ]:
summary_path = OUT / 'city_statistics.csv'
city_lookup.to_csv(summary_path, index=False, encoding='utf-8-sig')
restored_summary = pd.read_csv(summary_path, encoding='utf-8-sig', dtype={'city': 'string'})
print('保存位置：', summary_path.relative_to(ROOT))
display(restored_summary.head(3))
pd.testing.assert_frame_equal(restored_summary, city_lookup, check_exact=False, check_dtype=False)

Cookbook 的“从 URL 读 CSV”用法是 `pd.read_csv(url)`，前提是响应确实是可访问的 CSV，
而不是登录页或普通 HTML。它不会自动把任意网页转换成表。
在线采集流程见 [Weibo Crawler](01_weibo_crawler.ipynb)。
[官方说明：组合多张表](https://pandas.pydata.org/docs/getting_started/intro_tutorials/08_combine_dataframes.html)

<a id="homework5"></a>
### 第 5 部分作业：拼接、连接并保存城市汇总

本部分已有 `first_week`、`second_week` 和城市汇总 `city_lookup`。请完成：

1. 把两段记录上下拼成 `p5_posts`，重新生成行索引。
2. 取其中前 10 条的 `post_id` 与 `city`，左连接 `city_lookup`，保存为 `p5_joined`。
   检查右表城市键唯一，并显示连接指示列。连接后应保留这 10 行。
3. 只将城市汇总 `city_lookup` 保存到 `OUT / 'exercise5_city_lookup.csv'`，不保存帖子正文。
   使用适合中文的编码，不把索引另写成一列，再读回显示前 3 行。

说明 `concat` 与 `merge` 在这个例子中各解决什么问题。

**请在下面的空白 Python 代码格中完成；参考答案紧接在其后，默认折叠。**

<details><summary>展开第 5 部分参考答案与完整代码</summary><p><strong>参考思路：</strong> `concat` 按行拼接同结构记录，`merge` 按城市键补上汇总信息。 左连接保留左表记录；`validate=&#x27;many_to_one&#x27;` 检查右表键唯一，避免意外增加行数。</p><p>`both` 表示连接成功。`index=False` 不保存额外索引列，`utf-8-sig` 便于部分 Excel 版本识别中文。 CSV 不能完整保留 Pandas 的内部类型，因此回读时指定城市为字符串；该城市汇总仍是城市指标。</p><pre><code class="language-python">p5_posts = pd.concat([first_week, second_week], ignore_index=True)
p5_joined = p5_posts[[&#x27;post_id&#x27;, &#x27;city&#x27;]].head(10).merge(
    city_lookup, on=&#x27;city&#x27;, how=&#x27;left&#x27;, validate=&#x27;many_to_one&#x27;, indicator=True)
display(p5_joined)
print(&#x27;拼接行数：&#x27;, len(p5_posts), &#x27;连接行数：&#x27;, len(p5_joined))
p5_summary_path = OUT / &#x27;exercise5_city_lookup.csv&#x27;
city_lookup.to_csv(p5_summary_path, index=False, encoding=&#x27;utf-8-sig&#x27;)
p5_restored = pd.read_csv(p5_summary_path, encoding=&#x27;utf-8-sig&#x27;, dtype={&#x27;city&#x27;: &#x27;string&#x27;})
display(p5_restored.head(3))
print(&#x27;汇总保存位置：&#x27;, p5_summary_path.relative_to(ROOT))</code></pre></details>

<a id="chapter6"></a>
## 6．字符串与重采样：讨论内容随日期怎样变化？

### 6.1 str：逐条处理字符串

`.str.len()` 求每条文本的长度，`.str.contains()` 检查是否含有某个模式，
`.str.replace()` 替换子串。它们可直接作用于一列，不必自己写逐行循环。
`regex=False` 表示字面匹配；`regex=True` 时模式中的 `|` 表示“或”。
字符串长度是字符数，不是中文词数。

In [ ]:
text_examples = df[['text']].head(3).copy()
text_examples['字符数'] = text_examples['text'].str.len()
text_examples['前12个字符'] = text_examples['text'].str.slice(0, 12)
text_examples['去首尾空白'] = text_examples['text'].str.strip()
display(text_examples)

literal_food = df['text'].str.contains('火锅', regex=False, na=False)
any_food = df['text'].str.contains('火锅|美食|吃饭', regex=True, na=False)
print('含“火锅”的记录数：', literal_food.sum())
print('含任意一个示范饮食词的记录数：', any_food.sum())
assert literal_food.sum() <= any_food.sum()

**怎么看结果：** 多词匹配扩大了覆盖范围，但也可能包含广告、引用或其他语境。
包含某个词不一定表示作者正在从事相应行为。若词本身含 `.`、`+`、`(` 等字符，
字面匹配可避免把它们误当成正则语法。

### 6.2 resample：按时间间隔重新汇总

`set_index('created_at')` 将发帖时间变成索引；`resample('D')` 按本地日历日分组。
对布尔列求 `sum` 得到命中数，求 `mean` 得到非空布尔值中 True 的比例。
Cookbook 按月统计天气，这里用两周帖子按日演示，避免将短窗口误当成长时间序列。

In [ ]:
text_timeline = df.assign(has_food=any_food).set_index('created_at').sort_index()
daily = text_timeline.resample('D').agg(
    n_posts=('post_id', 'size'),
    n_food=('has_food', 'sum'),
    food_share=('has_food', 'mean'),
)
display(daily.head(5))
assert daily['n_posts'].sum() == len(df)
assert daily.loc[daily['n_posts'].gt(0), 'food_share'].between(0, 1).all()
fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
daily['n_posts'].plot(ax=axes[0], marker='o')
daily['food_share'].plot(ax=axes[1], marker='o')
axes[0].set_ylabel('采集记录数' if CJK_FONT else 'Collected posts')
axes[1].set_ylabel('饮食词命中比例' if CJK_FONT else 'Food keyword share')
axes[1].set_xlabel('上海时区日期' if CJK_FONT else 'Date (Asia/Shanghai)')
fig.suptitle('每日数量与比例分别看' if CJK_FONT else 'Daily counts and proportions')
plt.tight_layout()
plt.show()

**怎么看结果：** 上图是数量，下图是比例；两者的单位不同，所以分开画。
同一天的命中数除以该日全部记录数，才是该日命中比例。
空时间段的计数可以表示没有采集记录；其比例不能在没有分母时补成 0。
采集量变化不一定是讨论热度变化，也可能来自采集覆盖变化。

<a id="homework6"></a>
### 第 6 部分作业：从学习相关词生成每日统计表

检查正文是否包含“学习”或“考试”中的任意一个词，生成 `p6_has_study`。
按上海日历日汇总为 `p6_daily`，包含：

- `n_posts`：每日全部采集记录数。
- `n_study`：每日关键词命中记录数。
- `study_share`：每日命中比例。

显示前 5 行。解释对布尔列求 `sum` 和 `mean` 的区别，以及为什么没有记录的日期不能把比例补为 0。
使用本部分的 `.str.contains`、时间索引与 `.resample('D')`。

**请在下面的空白 Python 代码格中完成；参考答案紧接在其后，默认折叠。**

<details><summary>展开第 6 部分参考答案与完整代码</summary><p><strong>参考思路：</strong> 正则中的 `|` 表示“或”；缺失文本按不命中处理。 先新增布尔列，再将发帖时间设为索引并按日重采样。</p><p>布尔列 `sum` 计算 True 数量，`mean` 计算有效布尔值中的 True 比例。 分母是当日全部记录数。空时间段没有有效分母，比例应保持缺失。 词命中只是文本规则，仍不能说明作者实际在学习。</p><pre><code class="language-python">p6_has_study = df[&#x27;text&#x27;].str.contains(&#x27;学习|考试&#x27;, regex=True, na=False)
p6_timeline = df.assign(has_study=p6_has_study).set_index(&#x27;created_at&#x27;).sort_index()
p6_daily = p6_timeline.resample(&#x27;D&#x27;).agg(
    n_posts=(&#x27;post_id&#x27;, &#x27;size&#x27;),
    n_study=(&#x27;has_study&#x27;, &#x27;sum&#x27;),
    study_share=(&#x27;has_study&#x27;, &#x27;mean&#x27;),
)
display(p6_daily.head(5))</code></pre></details>

<a id="chapter7"></a>
## 7．清洗：识别缺失值、脏字符串与重复记录

### 7.1 先查真实表，再决定清洗规则

`.isna().sum()` 逐列数缺失值；`.duplicated('post_id')` 按帖子 ID 检查重复。
前面读入的记录已经过质量筛选；下面用微型表格说明常见脏值的处理方法。

In [ ]:
display(df[columns].isna().sum().to_frame('缺失数量'))
print('重复帖子 ID 数量：', df.duplicated('post_id').sum())
print('清理首尾空白后为空的正文数量：', df['text'].str.strip().eq('').sum())
print('city_code 中不同编码数：', df['city_code'].nunique())

### 7.2 微型教学表：让问题看得见

下面的五行**专门为解释清洗操作构造，不是真实帖子或地区统计**。
ID 与编码是字符串；`00123` 的前导零有意义。分数列中的空白、未知和坏字符不能直接当成 0。
这个对象独立命名为 `dirty_demo`，不写回真实 `df` 或源文件。

In [ ]:
dirty_demo = pd.DataFrame({
    '记录ID': ['a', 'b', 'b', 'c', 'd'],
    '编码': ['00123', '00124', '00124', 'UNKNOWN', None],
    '标签': [' 城市甲 ', '城市乙', '城市乙', '', None],
    '分数': ['0.8', '0.5', '0.5', '未知', '坏字符'],
})
display(dirty_demo)
clean_demo = dirty_demo.copy()
clean_demo['编码'] = clean_demo['编码'].astype('string').replace('UNKNOWN', pd.NA)
clean_demo['标签'] = clean_demo['标签'].astype('string').str.strip().replace('', pd.NA)
clean_demo['分数'] = pd.to_numeric(clean_demo['分数'], errors='coerce')
display(clean_demo)
display(clean_demo.isna().sum().to_frame('清洗后缺失数量'))
assert clean_demo.loc[0, '编码'] == '00123'
assert clean_demo['分数'].isna().sum() == 2

`errors='coerce'` 将无法解析的数字转成缺失值，**不是纠正了坏字符**；实际项目应记录转换失败数。
`None`、`pd.NA`、`NaN` 等可用 `isna()` 统一识别；空字符串默认不是缺失，需要另行处理。
`astype('string')` 使用可表达缺失值的字符串类型，避免把空值变成字面字符串 `'nan'`。

### 7.3 去重、删除缺失与补值分别解决什么问题？

`drop_duplicates` 需要有业务依据。教学表中的重复 ID 行完全相同，演示保留一行；
若同 ID 的内容冲突，应先核查版本规则。不能按正文去重真实帖子，不同用户可能发相同文字。
`dropna(subset=...)` 只根据指定必需字段删除；`fillna` 只在有合理语义时补值。

In [ ]:
unique_demo = clean_demo.drop_duplicates(subset='记录ID', keep='first')
known_code_demo = unique_demo.dropna(subset=['编码'])
print('原行数、去重后行数、有编码行数：', len(clean_demo), len(unique_demo), len(known_code_demo))
display(known_code_demo)
display(unique_demo['标签'].fillna('未记录').to_frame('仅用于显示的标签'))

count_demo = clean_demo.groupby('标签', dropna=False).agg(
    行数=('分数', 'size'), 有效分数数=('分数', 'count'))
display(count_demo)
assert count_demo['行数'].sum() == len(clean_demo)
assert count_demo['有效分数数'].sum() == clean_demo['分数'].notna().sum()

**怎么看结果：** 缺失标签所在组有记录，但可能没有有效分数，所以 `size` 与 `count` 不一定相等。
“未记录”适合展示未知标签；缺失分数仍保持缺失，不能替换成“中性”、零分或其他结论。
真实坐标缺失也不能补成经纬度 `(0, 0)`。
[官方说明：缺失数据](https://pandas.pydata.org/docs/user_guide/missing_data.html)

<a id="homework7"></a>
### 第 7 部分作业：清洗教学表，保留未知信息

从本部分的微型教学表 `dirty_demo` 重新复制一份 `p7_clean`，完成：

1. 把“编码”转成可表达缺失值的字符串类型，将 `UNKNOWN` 设为缺失，保留 `00123` 的前导零。
2. 去掉“标签”首尾空白，并把空字符串设为缺失。
3. 把“分数”转成数值，无法解析的内容设为缺失，不填成 0。
4. 按“记录ID”去重，保留第一行；本教学表重复 ID 的行内容完全相同。
   保留编码未知的记录，显示清洗结果和各列缺失数量。

解释为什么这次不能用无参数的 `dropna()` 删除所有含缺失值的行。

**请在下面的空白 Python 代码格中完成；参考答案紧接在其后，默认折叠。**

<details><summary>展开第 7 部分参考答案与完整代码</summary><p><strong>参考思路：</strong> 清洗规则分别针对编码、标签和分数，不能对所有列采取同一种补值办法。 本教学表按记录 ID 去重后保留 4 行，未知编码和不可解析的分数仍保留缺失。</p><p>缺编码或缺分数不一定使整条记录失去分析价值。直接 `dropna()` 会把这些记录删掉， 改变数据范围；应该根据当前问题决定必需字段。示例只处理教学表，不修改真实微博源。</p><pre><code class="language-python">p7_clean = dirty_demo.copy()
p7_clean[&#x27;编码&#x27;] = p7_clean[&#x27;编码&#x27;].astype(&#x27;string&#x27;).replace(&#x27;UNKNOWN&#x27;, pd.NA)
p7_clean[&#x27;标签&#x27;] = p7_clean[&#x27;标签&#x27;].astype(&#x27;string&#x27;).str.strip().replace(&#x27;&#x27;, pd.NA)
p7_clean[&#x27;分数&#x27;] = pd.to_numeric(p7_clean[&#x27;分数&#x27;], errors=&#x27;coerce&#x27;)
p7_clean = p7_clean.drop_duplicates(subset=&#x27;记录ID&#x27;, keep=&#x27;first&#x27;)
display(p7_clean)
display(p7_clean.isna().sum().to_frame(&#x27;缺失数量&#x27;))</code></pre></details>

<a id="chapter8"></a>
## 8．时间：字符串、Unix 时间戳和本地时区

### 8.1 日期字符串要按已知格式解析

字符串只是文字；时间类型才支持 `.dt.hour` 等操作。
本数据的 `date` 格式明确为年-月-日，可按 `%Y-%m-%d` 解析。
缺时区的字符串先确认它表达哪个地区的本地时间，再考虑 `tz_localize`。

In [ ]:
dates_from_text = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='raise')
time_columns = df[['created_at']].head(3).copy()
time_columns['本地小时'] = time_columns['created_at'].dt.hour
time_columns['星期编号'] = time_columns['created_at'].dt.dayofweek
time_columns['本地日期'] = time_columns['created_at'].dt.strftime('%Y-%m-%d')
display(time_columns)
assert dates_from_text.dt.strftime('%Y-%m-%d').eq(df['date']).all()
assert df['created_at'].dt.strftime('%Y-%m-%d').eq(df['date']).all()

### 8.2 Unix 时间戳：数字的单位和时区都要明确

Unix 时间戳表示从 UTC 的 1970-01-01 起经过的时间。
`unit='s'` 表示秒；毫秒来源应使用 `unit='ms'`。单位写错会得到错误时间。
以下两个整数是时间转换示例，没有对应任何新增帖子。

In [ ]:
unix_seconds = pd.Series([1660262400, 1660266000], name='Unix秒')
utc_times = pd.to_datetime(unix_seconds, unit='s', utc=True)
shanghai_times = utc_times.dt.tz_convert('Asia/Shanghai')
display(pd.DataFrame({'Unix秒': unix_seconds, 'UTC时间': utc_times,
                      '上海时间': shanghai_times}))
# 同一时间点的小时显示不同：分析本地行为时要先转换时区，再提取小时。
print('UTC 小时：', utc_times.dt.hour.tolist())
print('上海小时：', shanghai_times.dt.hour.tolist())
assert shanghai_times.iloc[1] - shanghai_times.iloc[0] == pd.Timedelta(hours=1)

**怎么看结果：** `tz_convert` 改变同一时间点的时区显示，不改变实际时间点。
`tz_localize` 给原本不带时区的时间附上已确认的时区，二者用途不同。
开场对真实 `created_at` 使用 `utc=True`，因为源字段已经是带时区的时间；
不能把来源不明的本地时间字符串直接假定成 UTC。

### 8.3 时间范围、排序和小时统计

时间筛选可用“左闭右开”的区间，避免末日只取到午夜。
`hour` 编号按 0 到 23 补齐，才能得到顺序正确的小时统计表。

In [ ]:
start = pd.Timestamp('2022-08-12', tz='Asia/Shanghai')
stop = pd.Timestamp('2022-08-19', tz='Asia/Shanghai')
week_one = df.loc[(df['created_at'] >= start) & (df['created_at'] < stop)]
hourly_counts = df.groupby(df['created_at'].dt.hour).size().reindex(range(24), fill_value=0)
print('8 月 12 日起、8 月 19 日前的记录数：', len(week_one))
display(hourly_counts.rename_axis('本地小时').rename('记录数').to_frame().head(8))
assert hourly_counts.sum() == len(df)

小时记录少，表示采集到的发帖记录少，不能直接推断用户在睡觉、离线或未使用手机。
`resample('D')` 将时间划成日历区间，`groupby(.dt.hour)` 则把不同日期的同一小时放在一起。
[官方说明：时间序列](https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html)

<a id="homework8"></a>
### 第 8 部分作业：转换 Unix 时间，并筛选上海的一整天

使用本部分已有的 `unix_seconds`：

1. 按秒解析为 UTC 时间 `p8_utc`，再转换为上海时间 `p8_shanghai`。
2. 生成 `p8_times`，并排显示 Unix 秒、UTC 时间、上海时间和上海小时。
3. 从真实帖子表 `df` 中选出上海日历日期 2022-08-12 的全部记录，命名为 `p8_day`。
   使用“当日零点及以后、次日零点之前”的区间，并显示数量。

解释为什么转换时区后小时不同，但实际时间点没有改变；为什么不能只用 `<= '2022-08-12'` 筛选整天。

**请在下面的空白 Python 代码格中完成；参考答案紧接在其后，默认折叠。**

<details><summary>展开第 8 部分参考答案与完整代码</summary><p><strong>参考思路：</strong> Unix 秒按 `unit=&#x27;s&#x27;` 且 `utc=True` 解析，再用 `tz_convert` 转换显示时区。 示例的 UTC 小时是 0、1，上海小时是 8、9，但对应同一组实际时间点。</p><p>用明确带上海时区的起止时间筛选，左闭右开可覆盖整天，且不包含次日零点。 只比较某个日期的午夜上界，可能漏掉当日零点之后的记录。 来源不明的无时区字符串不能直接假定为 UTC，毫秒时间戳也不能按秒解析。</p><pre><code class="language-python">p8_utc = pd.to_datetime(unix_seconds, unit=&#x27;s&#x27;, utc=True)
p8_shanghai = p8_utc.dt.tz_convert(&#x27;Asia/Shanghai&#x27;)
p8_times = pd.DataFrame({
    &#x27;Unix秒&#x27;: unix_seconds, &#x27;UTC时间&#x27;: p8_utc,
    &#x27;上海时间&#x27;: p8_shanghai, &#x27;上海小时&#x27;: p8_shanghai.dt.hour,
})
display(p8_times)
p8_start = pd.Timestamp(&#x27;2022-08-12&#x27;, tz=&#x27;Asia/Shanghai&#x27;)
p8_stop = pd.Timestamp(&#x27;2022-08-13&#x27;, tz=&#x27;Asia/Shanghai&#x27;)
p8_day = df.loc[(df[&#x27;created_at&#x27;] &gt;= p8_start) &amp; (df[&#x27;created_at&#x27;] &lt; p8_stop)].copy()
print(&#x27;上海 2022-08-12 的记录数：&#x27;, len(p8_day))
display(p8_day[[&#x27;created_at&#x27;, &#x27;city&#x27;, &#x27;text&#x27;]].head(3))</code></pre></details>

<a id="chapter9"></a>
## 9．SQL：从数据库读表、把表写入数据库

### 9.1 SQLite 也可以作为表格来源

SQLite 随 Python 标准库提供，无需额外服务器、账号或密码。
我们从真实源中取 30 行，在**内存数据库**中创建表，完成读写演示后关闭连接。
演示只选匿名帖子 ID、城市和日期，不将正文写入数据库。

In [ ]:
sql_sample = df[['post_id', 'city', 'date']].head(30).copy()
with sqlite3.connect(':memory:') as connection:
    sql_sample.to_sql('posts_demo', connection, index=False, if_exists='fail')
    sql_head = pd.read_sql_query('SELECT post_id, city, date FROM posts_demo LIMIT 3', connection)
    sql_city = pd.read_sql_query(
        'SELECT city, COUNT(*) AS n_posts FROM posts_demo GROUP BY city ORDER BY city',
        connection, dtype={'city': 'string'})
    demo_city = sql_sample['city'].iloc[0]
    sql_selected = pd.read_sql_query(
        'SELECT post_id, city, date FROM posts_demo WHERE city = ?',
        connection, params=(demo_city,))

display(sql_head)
display(sql_city)
print('参数化查询的城市与记录数：', demo_city, len(sql_selected))
assert sql_city['n_posts'].sum() == len(sql_sample)

**怎么看结果：** `to_sql` 把 DataFrame 写成数据库表；`read_sql_query` 执行查询并将结果读成 DataFrame。
`COUNT(*)` 数行数，对应这个例子中的 `groupby(...).size()`。
`LIMIT 3` 没有 `ORDER BY` 时不承诺时间顺序，只用于展示少量行。
用 `params` 传查询值，避免把用户输入直接拼进 SQL 字符串。
`if_exists='fail'` 在同名表已存在时停止；其他选项如 `replace` 会替换表，使用前要明确用途。

### 9.2 用 Pandas 对照同一个汇总

数据库中的 30 行只是演示子集，不等于全课的 54,609 行统计范围。
比较 SQL 和 Pandas 时，先保证数据范围、键、缺失值规则与排序一致。

In [ ]:
pandas_city = sql_sample.groupby('city').size().rename('n_posts').reset_index().sort_values('city')
pd.testing.assert_frame_equal(sql_city.reset_index(drop=True),
                              pandas_city.reset_index(drop=True), check_dtype=False)
print('同一 30 行子集：SQL 与 Pandas 的城市计数一致。')

PostgreSQL、MySQL 等数据库通常通过 SQLAlchemy 和相应驱动连接，再交给 Pandas 读写。
它们需要额外环境与连接配置，本教程不把账号、密码写入 Notebook。
[官方说明：SQL 读写](https://pandas.pydata.org/docs/user_guide/io.html#sql-queries)

<a id="reference"></a>
## 操作速查：将一条分析线串起来

读入文件 → 检查行列、类型和缺失 → 选取需要的行列 → 新增解释明确的字段 →
按用户、城市或时间汇总 → 确认分母 → 显示表格、绘图与保存。

| 想做什么 | 常用写法 | 需要看清什么 |
|---|---|---|
| 看数据规模 | `df.shape`、`df.head(3)`、`df.info()` | head 只显示，不截断原表 |
| 取列 | `df['text']`、`df[['city', 'text']]` | Series 与 DataFrame |
| 取行 | `df.iloc[:3]`、`df.loc[mask, columns]` | 位置、标签与布尔条件 |
| 数类别 | `series.value_counts(dropna=False)` | 默认缺失处理与排序 |
| 加字段 | `df.assign(...)`、`df.loc[mask, 'col'] = value` | 避免链式赋值 |
| 做比例 | `分子 / 分母` | 标签对齐与零分母 |
| 分组 | `df.groupby('user_id').agg(...)` | size、count、nunique、mean 的差异 |
| 整理表 | `.sort_values(...)`、`.reset_index()`、`.rename(...)` | 索引和普通列不同 |
| 组合表 | `pd.concat(...)`、`left.merge(right, ...)` | 按索引拼接或按键匹配 |
| 处理文本 | `.str.strip()`、`.str.len()`、`.str.contains(...)` | 缺失、字面匹配、正则语法 |
| 处理缺失 | `.isna()`、`.dropna(subset=...)`、`.fillna(...)` | 先解释缺失含义，再确定规则 |
| 处理时间 | `pd.to_datetime(...)`、`.dt.hour`、`.resample('D')` | 格式、时间单位与时区 |
| 导出 | `.to_csv(..., index=False)`、`.to_sql(...)` | 编码、索引与写入位置 |

### 查阅 API

[Pandas 入门](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html)、
[索引与赋值](https://pandas.pydata.org/docs/user_guide/indexing.html)、
[表格合并](https://pandas.pydata.org/docs/user_guide/merging.html)。